# Notebook 02 — LLMs as Research Instruments



**Duration:** ~20 minutes



In notebook 01 you set up your tools and confirmed that the university machine is ready. In this notebook you send your first message to the large language model (LLM). Then we look at the model more carefully: what does it already know, where does its knowledge break down, and why does it sometimes give different answers to the same question?



An LLM is a system trained on a very large amount of text. It learns patterns from that text — facts, grammar, reasoning styles — and uses those patterns to generate new text. It does not look anything up. It does not search the internet. Everything it produces comes from patterns it absorbed during training.



This makes it useful, but also unreliable in specific ways. By the end of this notebook you will understand those limitations well enough to work with them.

## Setup

Run this cell first. It loads your base URL and API key from the `.env` file you set up in notebook 01.

In [73]:
# ============================================================
# SETUP CELL — Run this once at the start of every notebook
# ============================================================

import os, json, base64, requests, io
from dotenv import load_dotenv
from openai import OpenAI
from lxml import etree
from PIL import Image
from IPython.display import Image as IPImage, display

# Load your base URL and API key from the .env file (set up in notebook 01)
load_dotenv(override=True)
if not os.getenv("LLM_BASE_URL") or not os.getenv("LLM_API_KEY"):
    raise RuntimeError("Could not find LLM_BASE_URL or LLM_API_KEY. Check your .env file (see notebook 01).")

client = OpenAI(base_url=os.environ["LLM_BASE_URL"], api_key=os.environ["LLM_API_KEY"])

# Each Dell Pro Max machine serves one model. This is its name.
# It reads both text and images.
MODEL = "nvidia/Nemotron-3-Nano-Omni-30B-A3B-Reasoning-NVFP4"

# Turn off the model's "thinking" step. Answers come back several times faster,
# which matters when many people share one machine.
NO_THINKING = {"chat_template_kwargs": {"enable_thinking": False}}

print(f"Using model {MODEL}.")
print("Setup complete.")

Using model nvidia/Nemotron-3-Nano-Omni-30B-A3B-Reasoning-NVFP4.
Setup complete.


## Exercise a — Your first message to the model

Let's send a short message to the model and read its reply. This is the simplest possible call: we tell it which model to use, and what to say.

We will then go through the syntax to understand how it works.

Run the cell below.


In [74]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": "What is the capital of France?"}
    ]
)

print(response.choices[0].message.content)
print(os.environ["LLM_BASE_URL"])



The capital of France is Paris.
https://lais11.cer.auckland.ac.nz/v1


You should see `Paris`, probably in a short sentence. The exact wording may differ from your neighbour's — that is expected. We will explore why later in this notebook.

<details>
<summary><strong>Let's break down what just happened</strong></summary>

| Line | What it does |
|------|--------------|
| `response` | Stores the entire response from the API |
| `client.chat.completions.create(...)` | Sends a message to the LLM on the Dell Pro Max |
| `model=MODEL` | Tells it which model to use (we set this in the setup cell) |
| `messages=[{...}]` | The conversation — a list of messages, each with a `role` and `content` |
| `"role": "user"` | Labels who is speaking — `"user"` means you, the human asking the question |
| `"content": "..."` | The actual text you are sending |
| `response.choices[0].message.content` | Extracts the model's reply from the response |

You will see this pattern in every notebook. The only thing that changes is the message you send.

</details>


### Your turn

Change the question below to anything you like and run the cell. Try asking it something related to your own research area.

This cell has one addition: `extra_body=NO_THINKING`. Some LLMs, including ours, write hidden working notes before they answer. This is called "thinking". It makes answers slower, which matters when many people share one machine. `NO_THINKING` was defined in the setup cell and switches it off. From here on, every call in the workshop includes it.


In [64]:
# Change the question to anything you like.
# Some ideas:
#   "What is thematic analysis in qualitative research?"
#   "Summarise the purpose of the NZ Human Rights Act 1993 in two sentences."
#   "What are the ethical risks of using AI in social science research?"

my_question = "What are the ethical risks of using AI in social science research?"

response = client.chat.completions.create(
    model=MODEL,
    extra_body=NO_THINKING,   # switch off the model's hidden "thinking" step
    max_tokens = 500,              # limit the length of the answer
    messages=[
        {"role": "user", "content": my_question}
    ]
)

print(response.choices[0].message.content)



Using AI in social science research can bring numerous benefits, such as increased efficiency, accuracy, and the ability to analyze large datasets. However, it also introduces several ethical risks that researchers must consider and address. Here are some of the key ethical risks:

1. **Privacy and Confidentiality**: AI systems often require large amounts of data, which may include sensitive personal information. Ensuring the privacy and confidentiality of research participants is crucial. Researchers must obtain informed consent, anonymize data, and use secure data storage and processing methods to protect participants' privacy.

2. **Informed Consent**: AI algorithms can be complex and difficult to understand, making it challenging for participants to provide truly informed consent. Researchers must ensure that participants understand how their data will be used, the potential risks and benefits, and their rights regarding data access, correction, and deletion.

3. **Bias and Discri

## What does the model already know?

The model was trained on a broad range of text — books, websites, legal documents, academic papers. That means it has **implicit knowledge**: things it absorbed during training, without being given a specific source to read.

This is useful. But it is also risky, because you cannot tell which facts it learned accurately and which ones it reconstructed imperfectly.

Let's test this. We will ask the model two questions, one about something very well known and one about something obscure — without giving it any source text to read.

## Exercise b — A well-known topic

Why the sky is blue is explained in countless textbooks, websites and articles. The model has almost certainly seen the explanation many times during training.

Run the cell below. We are not giving the model any data — just asking what it already knows.

In [ ]:
response = client.chat.completions.create(
    model=MODEL,
    extra_body=NO_THINKING,
    messages=[
        {"role": "user", "content": "In a single sentence, explain why the sky is blue"}
    ]
)

print(response.choices[0].message.content)

You should see a confident, specific answer. The model sounds like it knows this topic well.

Now let's try something less well-known.

## Exercise c — An obscure Act

The Impounding Act 1955 is a real New Zealand Act that is still in force. It deals with a very specific (and quite old-fashioned) area of law. It is unlikely to appear in many modern training documents.

We will ask the model two questions about it — without giving it any source text. See how it goes.

In [21]:
response = client.chat.completions.create(
    model=MODEL,
    extra_body=NO_THINKING,
    messages=[
        {"role": "user", "content": "What is the main purpose of New Zealand's Impoundment Act 1955?"}
    ]
)

print(response.choices[0].message.content)


The main purpose of New Zealand's Impoundment Act 1955 is to regulate the impoundment of animals, particularly livestock, to ensure their welfare and to facilitate the resolution of ownership disputes. The Act provides a framework for the seizure, detention, and sale of impounded animals, as well as the responsibilities of those who impound animals and those who claim ownership.


In [22]:
response = client.chat.completions.create(
    model=MODEL,
    extra_body=NO_THINKING,
    messages=[
        {"role": "user", "content": "New Zealand's Impoundment Act 1955, what is the maximum fine for not discosing the destruction of wandering livestock onto your property given in `Special remedies for trespass by pigs, goats, or poultry`?"}
    ]
)

print(response.choices[0].message.content)


The maximum fine for not disclosing the destruction of wandering livestock onto your property, as specified in the "Special remedies for trespass by pigs, goats, or poultry" under New Zealand's Impoundment Act 1955, is $100. This fine is intended to ensure compliance with the Act's requirements for reporting and addressing livestock trespass.


Sources:
- Impoundment ACT: https://www.legislation.govt.nz/act/public/1955/108/en/latest/#DLM293864
- Why the sky is blue (NASA): https://spaceplace.nasa.gov/blue-sky/en/

Compare the two answers:

1. Did the model sound equally confident about both topics?
2. Did it give specific, verifiable details about the Impoundment Act, or did it produce vague, plausible-sounding text?
3. Would you trust either answer without checking the source?

"Hallucinations"
What we've seen is **implicit knowledge** in action. The model does not tell you when it is guessing. It produces fluent text regardless of whether its information is accurate. For well-documented topics it is often right. For obscure ones it may **hallucinate** — generate confident-sounding text that is partially or entirely wrong.

This is why, in notebooks 03–05, we will always give the model **real source data** to work with rather than relying on what it already knows.

## Temperature

Why do we have different answers to the same question? The model is not a calculator that gives the same output for the same input. It is a text generator. It builds its answer one word at a time. At each step it has a list of possible next words, each with a **probability**, and it picks one.

A setting called **temperature** controls how adventurous that pick is. At `temperature=0` the model always picks the most probable word. As the temperature goes up, it picks less probable words more often.

The easiest way to understand temperature is to see it. We will ask the same question at two settings, starting with `temperature=0`.

The cell below adds two settings to the call you already know:

- `temperature=0` sets the temperature.
- `max_tokens=500` limits the length of the reply. A **token** is a small chunk of text, roughly a word or part of a word. The model stops writing when it reaches the limit. You will see why we need this in a moment.

Run the cell below.

In [ ]:
response = client.chat.completions.create(
    model=MODEL,
    extra_body=NO_THINKING,
    temperature=0,     # always pick the most probable word
    max_tokens=500,    # stop after 500 tokens at most
    messages=[
        {"role": "user", "content": "Explain what qualitative research is, in three sentences."}
    ]
)

print(response.choices[0].message.content)

You should see a clear answer in three sentences.

Now the same question at `temperature=1.8`. Nothing else changes. This cell takes longer to run, about 15 seconds.

In [ ]:
response = client.chat.completions.create(
    model=MODEL,
    extra_body=NO_THINKING,
    temperature=1.8,   # pick unlikely words much more often
    max_tokens=500,    # stop after 500 tokens at most
    messages=[
        {"role": "user", "content": "Explain what qualitative research is, in three sentences."}
    ]
)

print(response.choices[0].message.content)

The answer may start sensibly, but it soon falls apart into a jumble of word fragments, symbols and other languages.

Here is what happened. At a high temperature the model often picks an unlikely next word. Each new word is chosen to follow the words already written, so one odd word leads to another and the answer never recovers. The model also never decides it has finished. Without `max_tokens=500` it would keep writing for a very long time.

Nobody uses `temperature=1.8` for real work. We use it here because it makes the effect easy to see. At lower settings the effect is the same but much more subtle: the text still reads well, and what changes is which words, and sometimes which facts, the model chooses.

### Temperature and consistency

Different results each run can be an issue for research - which seeks the opposite.

| Temperature | Behaviour | Use case |
|-------------|-----------|----------|
| `0` | Always picks the most probable word. The output may be nearly identical every time. | Research tasks where consistency matters |
| `1.2` | Picks words with more randomness. The output varies between runs. | Wider range of topics / keyword searches |
| `1.8` | Picks unlikely words so often that the text stops making sense. | A demonstration only |

## Exercise d — Temperature and repeatability

We will now ask the model the **same question four times**:

- Twice at `temperature=0` (always picks the most probable word)
- Twice at `temperature=1.2` (more randomness)

We will do this for two questions: first one about the obscure Impounding Act 1955, then the very well-known question of why the sky is blue. The aim is to find out when the model gives repeatable answers, and whether a repeatable answer is the same thing as a correct one.

To avoid writing eight nearly identical cells, we use two small pieces of Python:

- A **function** called `ask`. It wraps the API call you have already seen, so we can send any question at any temperature in one line.
- A **`for` loop**. It repeats the same block of code for each value in a list. `for run in [1, 2]:` runs the block once with `run` set to `1`, then once with it set to `2`.

Run the cell below to define the function and the two questions. It does not call the model yet.


In [ ]:
def ask(prompt, temperature):
    """Send one question to the model and return its reply as text."""
    response = client.chat.completions.create(
        model=MODEL,
        extra_body=NO_THINKING,
        temperature=temperature,
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

# One question on an obscure topic, and one on a very well-known topic.
IMPOUNDING_PROMPT = (
    "In a single sentence, state the maximum fine under New Zealand's Impounding Act 1955 "
    "for leaving a gate open on a public road, and the section of the Act that sets it"
)

SKY_PROMPT = "In a single sentence, explain why the sky is blue"

print("Function and prompts defined.")


### Part 1 — The obscure Act

First, the Impounding Act 1955. We ask for two specific facts: the maximum fine for leaving a gate open on a public road, and the section of the Act that sets it. We ask the same question four times, split over two cells: two runs at `temperature=0`, then two runs at `temperature=1.2`.

**Temperature = 0.** Are the two answers the same as each other?


In [ ]:
for run in [1, 2]:
    print(f"=== Impounding Act 1955 | temperature = 0 | run {run} ===")
    print(ask(IMPOUNDING_PROMPT, 0))
    print()


**Temperature = 1.2.** Same question, two more runs. Do the two answers agree with each other **on the facts**, or only on the general topic?


In [ ]:
for run in [1, 2]:
    print(f"=== Impounding Act 1955 | temperature = 1.2 | run {run} ===")
    print(ask(IMPOUNDING_PROMPT, 1.2))
    print()


The two `temperature=0` answers should be identical, or very close. That looks reassuring. But look at the two `temperature=1.2` answers: they probably give different fines, different section numbers, or both. When the model is free to vary its answer, the *facts* vary, not just the wording. That is a sign that it is reconstructing plausible-sounding text rather than recalling something it learned well.

You can check these answers against the real Act: https://www.legislation.govt.nz/act/public/1955/108/en/latest/whole.html. Look for section 58, "Offences by other persons".

### Part 2 — A well-known topic

Now the same experiment with a question the model has seen answered many times in training: why the sky is blue.

**Temperature = 0.** Two runs.


In [50]:
for run in [1, 2]:
    print(f"=== Why the sky is blue | temperature = 0 | run {run} ===")
    print(ask(SKY_PROMPT, 0))
    print()


=== Why the sky is blue | temperature = 0 | run 1 ===

The sky appears blue because molecules in Earth's atmosphere scatter shorter-wavelength blue light from the sun more than longer-wavelength colors, making blue the dominant color we see.

=== Why the sky is blue | temperature = 0 | run 2 ===

The sky appears blue because molecules in Earth's atmosphere scatter shorter-wavelength blue light from the sun more than longer-wavelength colors, making blue the dominant color we see.



**Temperature = 1.2.** Two more runs. Compare the facts, not just the wording.


In [51]:
for run in [1, 2]:
    print(f"=== Why the sky is blue | temperature = 1.2 | run {run} ===")
    print(ask(SKY_PROMPT, 1.2))
    print()


=== Why the sky is blue | temperature = 1.2 | run 1 ===

The sky is blue because molecules in the air scatter blue light from the sun more than other colors, making blue light reach our eyes more prominently.

=== Why the sky is blue | temperature = 1.2 | run 2 ===

The sky appears blue because molecules in Earth's atmosphere scatter short-wavelength blue light more efficiently than longer wavelengths, a phenomenon known as Rayleigh scattering.



### What did you notice?

Compare the two experiments.

| | Impounding Act 1955 (obscure) | Why the sky is blue (well-known) |
|---|---|---|
| `temperature=0`, two runs | The same fine each time, in nearly the same words | Same words each time |
| `temperature=1.2`, two runs | Different fines and different section numbers | Different wording, but the same explanation |
| Are the answers correct? | Almost certainly not. The Act sets a maximum fine of $100, in section 58 | Yes |

Two lessons come out of this.

**Temperature controls repeatability, not accuracy.** At `temperature=0` the model gives the same answer, or very nearly the same, every time, whether or not that answer is right. For the Impounding Act, you probably got the same *wrong* fine twice. A repeatable result is not evidence that the model knows what it is talking about.

You may still see small differences at `temperature=0`, such as a different section number. The machine's arithmetic is not perfectly exact, so when the model rates two words as almost equally likely, either one can come out on top. This happens more often when the model is unsure of its facts.

**Consistency across `temperature=1.2` runs tells you something about training.** When the model has seen a topic many times in training, the underlying facts are stable and only the phrasing changes between runs. When it has barely seen the topic, each run produces a different plausible-sounding reconstruction. Why the sky is blue is explained in a very large number of documents. The Impounding Act is not.

For research this matters in both directions. If you used `temperature=0` to classify 200 legislative sections, you would get the same classification every run. That is good for reproducibility. But if the model does not know the material, you would reproduce the same mistakes 200 times. This is why, in notebooks 03 to 05, we give the model the real source text to work from rather than relying on what it remembers.

You can check the sky answer here: https://spaceplace.nasa.gov/blue-sky/en/


## Exercise e — Inconsistency versus creativity

So far we have treated `temperature=1.2` as a problem — a source of inconsistency to be avoided. But that is not the whole story. Higher temperature makes the model produce a wider spread of answers. For some research tasks, that spread is exactly what you want.

Imagine you have a passage of text and you want to come up with keywords to code it. A single run at `temperature=0` gives you one consistent answer — good for reproducibility, but you might miss useful alternatives. Running at `temperature=1.2` gives you a different menu of candidate keywords to consider.

**First, you try.** Read the passage below and pick **10 keywords** that you think capture the substantive concepts. Then we will ask the model to do the same at `temperature=0` and `temperature=1.2`, and look at the three lists side by side.

In [52]:
PRIVACY_ACT_PASSAGE = """
What is personal information and the Privacy Act? 

Personal information is “information about an identifiable individual”. It covers both information that is simply about a person (e.g. eye colour) and information that may also identify them (e.g. their name). The information does not need to name the individual, as long as they are identifiable in other ways, like through their home address.

The Privacy Act 2020 provides the rules in New Zealand for protecting personal information and puts responsibilities on agencies and organizations about how they must do that. For example, people have a right to know what information your agency holds about them and a right to ask you to correct it if they think it is wrong.
"""

print(PRIVACY_ACT_PASSAGE)

# Your turn — replace these with 10 keywords you pick from the passage.
my_keywords = [
    "____",
    "____",
    "____",
    "____",
    "____",
    "____",
    "____",
    "____",
    "____",
    "____",
]


What is personal information and the Privacy Act? 

Personal information is “information about an identifiable individual”. It covers both information that is simply about a person (e.g. eye colour) and information that may also identify them (e.g. their name). The information does not need to name the individual, as long as they are identifiable in other ways, like through their home address.

The Privacy Act 2020 provides the rules in New Zealand for protecting personal information and puts responsibilities on agencies and organizations about how they must do that. For example, people have a right to know what information your agency holds about them and a right to ask you to correct it if they think it is wrong.



In [53]:
keyword_prompt = (
    "Extract 10 keywords that capture the substantive concepts in this passage. "
    "Return ONLY a JSON array of lowercase strings, e.g. [\"consent\", \"disclosure\"]. "
    "No preamble. No code fences.\n\n"
    f"Passage:\n{PRIVACY_ACT_PASSAGE}"
)

def extract_keywords(temperature):
    response = client.chat.completions.create(
        model=MODEL,
        extra_body=NO_THINKING,
        temperature=temperature,
        messages=[{"role": "user", "content": keyword_prompt}]
    )
    raw = response.choices[0].message.content.strip()
    if raw.startswith("```"):
        raw = "\n".join(l for l in raw.splitlines() if not l.startswith("```")).strip()
    return [k.lower().strip() for k in json.loads(raw)]

temp0_keywords = extract_keywords(0)
temp1_keywords = extract_keywords(1.2)

print("Temperature = 0:", temp0_keywords)
print()
print("Temperature = 1.2:", temp1_keywords)

Temperature = 0: ['personal information', 'privacy act', 'identifiable individual', 'home address', 'privacy act 2020', 'new zealand', 'agencies', 'organizations', 'right to know', 'right to correct']

Temperature = 1.2: ['personal information', 'identifiable individual', 'eye colour', 'home address', 'privacy act 2020', 'new zealand', 'protecting personal information', 'agency responsibilities', 'right to know', 'right to correct']


In [54]:
# Side-by-side comparison of the three lists.
rows = max(len(my_keywords), len(temp0_keywords), len(temp1_keywords))

header = f"{'Your keywords':<25} {'Temp = 0':<25} {'Temp = 1.2':<25}"
print(header)
print("-" * len(header))
for i in range(rows):
    a = my_keywords[i]    if i < len(my_keywords)    else ""
    b = temp0_keywords[i] if i < len(temp0_keywords) else ""
    c = temp1_keywords[i] if i < len(temp1_keywords) else ""
    print(f"{a:<25} {b:<25} {c:<25}")

Your keywords             Temp = 0                  Temp = 1.2               
-----------------------------------------------------------------------------
____                      personal information      personal information     
____                      privacy act               identifiable individual  
____                      identifiable individual   eye colour               
____                      home address              home address             
____                      privacy act 2020          privacy act 2020         
____                      new zealand               new zealand              
____                      agencies                  protecting personal information
____                      organizations             agency responsibilities  
____                      right to know             right to know            
____                      right to correct          right to correct         


### When is higher temperature actually useful?

Compare the three lists. Your own picks, `temperature=0`, and `temperature=1.2` will overlap in places and diverge in others:

- At `temperature=0`, you get one consistent list. Great for reproducibility — run it again and you get the same keywords.
- At `temperature=1.2`, you get a different spread. Some framings show up here that `temperature=0` would never surface.

This flips the usual advice. For **coding and extraction** — where consistency is everything — stick with `temperature=0`. But for **exploration** — when you want a menu of possible framings, keywords, or interpretations to consider — running at `temperature=1.2` a few times can surface ideas a single low-temperature run would miss.

## What you accomplished

In this notebook you explored two factors that shape the quality and direction of LLM outputs:

| Factor | What you observed | Exercise |
|--------|-------------------|----------|
| **Training data** | The model was confident on a well-known topic (why the sky is blue)<br>but unreliable on the obscure Impounding Act.<br>Its accuracy depends on how much relevant,<br>recent data it was trained on. | b & c |
| **Temperature** | Low temperature gives reproducible answers, but reproducible<br>does not mean correct: the obscure Act was wrong the same way<br>twice (exercise d). Higher temperature surfaces a wider spread of<br>candidate framings and keywords (exercise e). | d & e |

The key insight: an LLM is a **research instrument**, not an oracle. Like any instrument, it needs to be calibrated (temperature) and pointed at real data (not just its implicit knowledge). When either of these factors is weak, the output suffers — and the model will not tell you.

This is why, in notebooks 03–05, we default to `temperature=0` for coding and extraction, and reach for higher temperature only when we are still deciding what to look for.

**Next up:** After the break, notebook 03 introduces exploratory analysis. You will build the same request five ways, see how each layer of prompt structure changes the output, and then replicate a published coding method on a passage of real NZ legislation.